# Session 9 — Statistics for Neuroscience

**Goal of this session:** turn "these two conditions look different" into a defensible claim.

*Python for Neuroscience, session 9 of 12.*

## Why this matters

Every plot in this course so far has been descriptive. It showed you what happened in the data you have.

A statistical test asks a harder question: if there were no real difference, how often would noise alone produce something this big? That is the last step before you are allowed to say anything about brains rather than about your sample.

## Two conditions

We stay with the same generator. Condition A has a stronger oscillation than condition B on average, but every trial gets its own amplitude drawn at random, so the two groups overlap.

That overlap is the point. Real data always overlaps.

In [ ]:
import numpy as np


def generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5,
                        freq=10.0, amplitude=1.0, seed=0):
    """A toy oscillatory signal: one sine wave plus white noise."""
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / sampling_rate)
    signal = amplitude * np.sin(2 * np.pi * freq * t)
    signal = signal + noise_level * rng.standard_normal(t.size)
    return t, signal

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n_trials = 30

amp_A = rng.normal(1.00, 0.25, n_trials)      # condition A, stronger on average
amp_B = rng.normal(0.75, 0.25, n_trials)      # condition B, weaker on average

A, B = [], []
for i in range(n_trials):
    _, sig_a = generate_toy_signal(duration=1.0, amplitude=max(amp_A[i], 0.05),
                                   noise_level=0.3, seed=i)
    _, sig_b = generate_toy_signal(duration=1.0, amplitude=max(amp_B[i], 0.05),
                                   noise_level=0.3, seed=300 + i)
    A.append(sig_a.std())                     # our measure: signal amplitude
    B.append(sig_b.std())

A = np.array(A)
B = np.array(B)
print(A.shape, B.shape)

## Descriptive statistics first

Always. The mean tells you where the group sits, the standard deviation tells you how spread out the trials are, and the standard error tells you how well you have pinned down the mean.

Standard error is the standard deviation divided by the square root of the sample size. It shrinks as you collect more data. Standard deviation does not.

In [ ]:
from scipy import stats


def describe(x, name):
    sem = x.std(ddof=1) / np.sqrt(len(x))
    print(f"{name}: n = {len(x)}, mean = {x.mean():.3f}, "
          f"SD = {x.std(ddof=1):.3f}, SEM = {sem:.3f}")


describe(A, "Condition A")
describe(B, "Condition B")
print(f"\ndifference in means: {A.mean() - B.mean():.3f}")

`ddof=1` matters. It tells NumPy you are estimating the standard deviation of a population from a sample, which is what you are almost always doing. Without it you get a number that is slightly too small.

## The t-test

An independent-samples t-test compares two groups of separate observations. It gives you a t statistic, which is the difference in means scaled by how noisy the data is, and a p value.

In [ ]:
t_stat, p_value = stats.ttest_ind(A, B)

print(f"t = {t_stat:.2f}")
print(f"p = {p_value:.4f}")

## What the p value actually says

It is the probability of seeing a difference at least this large *if the two conditions were truly identical*. That is all.

It is not the probability that your hypothesis is true. It is not the size of the effect. A tiny difference will produce a tiny p value if you collect enough trials, and that is a statement about your sample size, not about the brain.

So report the effect size next to it. Cohen's d expresses the difference in units of the pooled standard deviation, and it does not care how many trials you ran.

In [ ]:
pooled_sd = np.sqrt((A.var(ddof=1) + B.var(ddof=1)) / 2)
cohens_d = (A.mean() - B.mean()) / pooled_sd

print(f"Cohen's d = {cohens_d:.2f}")
print("rough convention: 0.2 small, 0.5 medium, 0.8 large")

And one habit that will keep you honest: decide what you are testing before you look. If you run twenty comparisons and report the one that came out at p = 0.04, you have found noise, and the p value you printed is meaningless. Correct for multiple comparisons, or pre-register, or say plainly that the analysis was exploratory.

## The figure

The test result belongs on the plot, next to the data it came from.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

df = pd.DataFrame({
    "condition": ["A"] * len(A) + ["B"] * len(B),
    "amplitude": np.concatenate([A, B]),
})

sns.set_theme(style="whitegrid", font_scale=1.2)
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sns.violinplot(data=df, x="condition", y="amplitude", hue="condition",
               palette=["#2b6cb0", "#a0aec0"], legend=False, inner=None, ax=ax)
sns.stripplot(data=df, x="condition", y="amplitude", color="black",
              size=4, alpha=0.6, ax=ax)

y = df["amplitude"].max() * 1.08
ax.plot([0, 1], [y, y], color="black", linewidth=1.2)
ax.text(0.5, y * 1.01, f"t = {t_stat:.2f}, p = {p_value:.3f}, d = {cohens_d:.2f}",
        ha="center", fontsize=12)
ax.set_ylim(top=y * 1.12)
ax.set_xlabel("condition")
ax.set_ylabel("signal amplitude (SD)")
ax.set_title("Amplitude differs between conditions")
plt.tight_layout()
plt.show()

Look at how much the two violins overlap even with a significant result. That picture is worth keeping in mind every time you read "significantly different" in an abstract.

## Try it yourself

Change `n_trials` to 10 and rerun everything. The means barely move and the p value falls apart. Then try 200. Watching the p value chase sample size is the fastest way to stop over-trusting it.

**Next session:** brains as networks.